In [0]:
%pip install openpyxl -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Directory config ---
current_dir = "/Workspace/Users/c837609466@colostate.edu/Team_23"
shared_dir = "/Workspace/Shared/Team23"

# --- Load fruit fly detections ---
df_fruit_fly = pd.read_parquet(os.path.join(current_dir, "df_fruit_fly.parquet"))
print(f"Fruit fly detections: {df_fruit_fly.shape[0]} rows, {df_fruit_fly.shape[1]} cols")

# --- Load international segments (raw) ---
df_segments = pd.read_parquet(os.path.join(current_dir, "df_segments.parquet"))
print(f"International segments: {df_segments.shape[0]} rows, {df_segments.shape[1]} cols")

# --- Load domestic segments ---
try:
    df_domestic = pd.read_parquet(os.path.join(current_dir, "df_domestic_segment.parquet"))
    print(f"Domestic segments: {df_domestic.shape[0]} rows, {df_domestic.shape[1]} cols")
except Exception as e:
    print(f"Could not load domestic data: {e}")

Fruit fly detections: 305 rows, 8 cols
International segments: 544082 rows, 48 cols
Domestic segments: 2408684 rows, 50 cols


In [0]:
# --- Relevant columns for fruit fly / flight spread-risk analysis ---
# Time + location + volume + carrier + cargo/risk indicators

segment_cols = [
    # Time
    "YEAR_WADS", "QUARTER_WADS", "MONTH_WADS",
    # Carrier
    "UNIQUE_CARRIER_NAME", "CARRIER_NAME",
    # Origin
    "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_COUNTRY", "ORIGIN_COUNTRY_NAME",
    "ORIGIN_LAT", "ORIGIN_LON",
    # Destination
    "DEST", "DEST_CITY_NAME", "DEST_COUNTRY", "DEST_COUNTRY_NAME",
    "DEST_LAT", "DEST_LON",
    # Volume & operations
    "DEPARTURES_PERFORMED", "SEATS", "PASSENGERS", "DISTANCE",
    # Cargo & risk — added back for fruit fly spread analysis
    "FREIGHT",   # agricultural cargo is a key vector for pest spread
    "REGION",    # geographic risk zone classification
    "CLASS",     # cargo vs passenger service type
]

# --- Trim international segments (re-save with added columns) ---
df_segments_trimmed = df_segments[segment_cols].copy()
out_path_intl = os.path.join(current_dir, "international_segments_trimmed.parquet")
df_segments_trimmed.to_parquet(out_path_intl, index=False)
print(f"International segments: {df_segments.shape[1]} -> {df_segments_trimmed.shape[1]} cols")
print(f"Saved to {out_path_intl}\n")

# --- Load and trim domestic segments ---
df_domestic = pd.read_parquet(os.path.join(current_dir, "df_domestic_segment.parquet"))
print(f"Domestic segments loaded: {df_domestic.shape[0]} rows, {df_domestic.shape[1]} columns")

# Use only columns that exist in domestic (some may differ)
dom_cols = [c for c in segment_cols if c in df_domestic.columns]
missing = set(segment_cols) - set(dom_cols)
if missing:
    print(f"Note: domestic data missing columns: {sorted(missing)}")

df_domestic_trimmed = df_domestic[dom_cols].copy()
out_path_dom = os.path.join(current_dir, "domestic_segments_trimmed.parquet")
df_domestic_trimmed.to_parquet(out_path_dom, index=False)
print(f"Domestic segments: {df_domestic.shape[1]} -> {df_domestic_trimmed.shape[1]} cols")
print(f"Saved to {out_path_dom}\n")

print("Kept columns:")
for c in segment_cols:
    marker = " (cargo/risk)" if c in ["FREIGHT", "REGION", "CLASS"] else ""
    print(f"  {c}{marker}")

International segments: 48 -> 24 cols
Saved to /Workspace/Users/c837609466@colostate.edu/Team_23/international_segments_trimmed.parquet

Domestic segments loaded: 2408684 rows, 50 columns
Note: domestic data missing columns: ['DEST_COUNTRY', 'DEST_COUNTRY_NAME', 'MONTH_WADS', 'ORIGIN_COUNTRY', 'ORIGIN_COUNTRY_NAME', 'QUARTER_WADS', 'YEAR_WADS']
Domestic segments: 50 -> 17 cols
Saved to /Workspace/Users/c837609466@colostate.edu/Team_23/domestic_segments_trimmed.parquet

Kept columns:
  YEAR_WADS
  QUARTER_WADS
  MONTH_WADS
  UNIQUE_CARRIER_NAME
  CARRIER_NAME
  ORIGIN
  ORIGIN_CITY_NAME
  ORIGIN_COUNTRY
  ORIGIN_COUNTRY_NAME
  ORIGIN_LAT
  ORIGIN_LON
  DEST
  DEST_CITY_NAME
  DEST_COUNTRY
  DEST_COUNTRY_NAME
  DEST_LAT
  DEST_LON
  DEPARTURES_PERFORMED
  SEATS
  PASSENGERS
  DISTANCE
  FREIGHT (cargo/risk)
  REGION (cargo/risk)
  CLASS (cargo/risk)


In [0]:
# Inspect the key fields we'll join on: time + geography

# --- Fruit fly time format ---
print("FRUIT FLY - monthyear samples:")
print(df_fruit_fly['monthyear'].unique()[:10])
print(f"  States: {sorted(df_fruit_fly['STATE_NAME'].unique())}")
print()

# --- International flights time + destination ---
df_intl = pd.read_parquet(os.path.join(current_dir, "international_segments_trimmed.parquet"))
print("INTERNATIONAL - time range:")
print(f"  Years: {sorted(df_intl['YEAR_WADS'].unique())}")
print(f"  Months: {sorted(df_intl['MONTH_WADS'].unique())}")
print()
print("INTERNATIONAL - DEST_CITY_NAME samples:")
print(df_intl['DEST_CITY_NAME'].drop_duplicates().head(20).tolist())
print()
print("INTERNATIONAL - DEST_COUNTRY samples:")
print(df_intl[df_intl['DEST_COUNTRY'] == 'US']['DEST_CITY_NAME'].drop_duplicates().head(20).tolist())

FRUIT FLY - monthyear samples:
['01/2025' '02/2022' '02/2024' '02/2025' '03/2022' '03/2024' '03/2025'
 '04/2022' '04/2024' '04/2025']
  States: ['California', 'Florida', 'Texas']

INTERNATIONAL - time range:
  Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

INTERNATIONAL - DEST_CITY_NAME samples:
['Porto, Portugal', 'Nassau, The Bahamas', 'Oslo, Norway', 'Munich, Germany', 'Port-au-Prince, Haiti', 'Nairobi, Kenya', 'Providenciales, Turks and Caicos Islands', 'Nice, France', 'Northolt, United Kingdom', 'Nagoya, Japan', 'Connaught, Ireland', 'Panama City, Panama', 'Santiago, Dominican Republic', 'Ponta Delgada, Portugal', 'Punta Cana, Dominican Republic', 'Maldonado, Uruguay', 'London, United Kingdom', 'Stuttgart, Germany', 'Beijing, China', 'Perpignan, France']

I

In [0]:
# ===================================================================
# OPTION A: County-level fruit fly joined with state-level flights
# Join grain: state + year + month (fruit fly keeps county detail)
# Flights broken down by ORIGIN COUNTRY
# ===================================================================

# --- State abbreviation -> full name mapping ---
state_abbr_to_name = {
    'TX': 'Texas', 'FL': 'Florida', 'CA': 'California',
    'AZ': 'Arizona', 'NY': 'New York', 'GA': 'Georgia',
    'IL': 'Illinois', 'NJ': 'New Jersey', 'VA': 'Virginia',
    'PA': 'Pennsylvania', 'MA': 'Massachusetts', 'DC': 'Washington DC',
    'HI': 'Hawaii', 'AK': 'Alaska', 'LA': 'Louisiana',
    'MN': 'Minnesota', 'CT': 'Connecticut', 'ME': 'Maine',
    'NC': 'North Carolina', 'SC': 'South Carolina', 'OH': 'Ohio',
    'MI': 'Michigan', 'WA': 'Washington', 'OR': 'Oregon',
    'CO': 'Colorado', 'NV': 'Nevada', 'TN': 'Tennessee',
    'MD': 'Maryland', 'MO': 'Missouri', 'IN': 'Indiana',
    'WI': 'Wisconsin', 'KY': 'Kentucky', 'AL': 'Alabama',
    'OK': 'Oklahoma', 'UT': 'Utah', 'NM': 'New Mexico',
    'MS': 'Mississippi', 'AR': 'Arkansas', 'IA': 'Iowa',
    'KS': 'Kansas', 'NE': 'Nebraska', 'ID': 'Idaho',
    'WV': 'West Virginia', 'NH': 'New Hampshire', 'MT': 'Montana',
    'RI': 'Rhode Island', 'DE': 'Delaware', 'SD': 'South Dakota',
    'ND': 'North Dakota', 'VT': 'Vermont', 'WY': 'Wyoming',
    'PR': 'Puerto Rico', 'VI': 'Virgin Islands', 'GU': 'Guam',
}

# ---- 1. Prepare fruit fly data (KEEP county-level) ----
df_ff = df_fruit_fly.copy()
df_ff['month'] = df_ff['monthyear'].str.split('/').str[0].astype(int)
df_ff['year'] = df_ff['monthyear'].str.split('/').str[1].astype(int)
df_ff = df_ff.rename(columns={
    'STATE_NAME': 'state',
    'NAME': 'county',
    'CommonName': 'fly_species',
    'Count_': 'fly_detections'
})
df_ff = df_ff[['state', 'county', 'year', 'month', 'fly_species', 'fly_detections']]
print(f"Fruit fly (county-level): {df_ff.shape[0]} rows")

# ---- 2. Prepare international flights (aggregate to state-month-country) ----
df_intl = pd.read_parquet(os.path.join(current_dir, "international_segments_trimmed.parquet"))
df_us = df_intl[df_intl['DEST_COUNTRY'] == 'US'].copy()
df_us['state_abbr'] = df_us['DEST_CITY_NAME'].str.extract(r',\s*([A-Z]{2})$')
df_us['state'] = df_us['state_abbr'].map(state_abbr_to_name)

# Group by state + month + ORIGIN COUNTRY (preserves country breakdown)
flight_agg = (
    df_us[df_us['state'].notna()]
    .groupby(['state', 'YEAR_WADS', 'MONTH_WADS', 'ORIGIN_COUNTRY_NAME'])
    .agg(intl_departures=('DEPARTURES_PERFORMED', 'sum'),
         intl_passengers=('PASSENGERS', 'sum'),
         intl_seats=('SEATS', 'sum'),
         intl_freight=('FREIGHT', 'sum'),
         avg_distance=('DISTANCE', 'mean'))
    .reset_index()
    .rename(columns={'YEAR_WADS': 'year', 'MONTH_WADS': 'month',
                     'ORIGIN_COUNTRY_NAME': 'origin_country'})
)
print(f"International flights (state-month-country): {flight_agg.shape[0]} rows")
print(f"  Unique origin countries: {flight_agg['origin_country'].nunique()}")

# ---- 3. Left join: county-level fly data gets state-level flight stats per origin country ----
df_option_a = pd.merge(
    df_ff, flight_agg,
    on=['state', 'year', 'month'],
    how='left'
).sort_values(['state', 'county', 'year', 'month', 'origin_country']).reset_index(drop=True)

print(f"\nOption A combined: {df_option_a.shape[0]} rows, {df_option_a.shape[1]} columns")
print(f"  Each fruit fly row is now split by origin country of inbound flights.")

# ---- 4. Save ----
out_path_a = os.path.join(current_dir, "option_a_county_state_combined.parquet")
df_option_a.to_parquet(out_path_a, index=False)
print(f"Saved to {out_path_a}")

# Preview: show Cameron County with country breakdown
print(f"\nSample \u2014 Cameron County, TX (Feb 2024, by origin country):")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(df_option_a[
    (df_option_a['county'] == 'Cameron County') &
    (df_option_a['year'] == 2024) &
    (df_option_a['month'] == 2)
])

Fruit fly (county-level): 305 rows
International flights (state-month-country): 50293 rows
  Unique origin countries: 169

Option A combined: 14106 rows, 12 columns
  Each fruit fly row is now split by origin country of inbound flights.
Saved to /Workspace/Users/c837609466@colostate.edu/Team_23/option_a_county_state_combined.parquet

Sample — Cameron County, TX (Feb 2024, by origin country):


state,county,year,month,fly_species,fly_detections,origin_country,intl_departures,intl_passengers,intl_seats,intl_freight,avg_distance
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Argentina,70.0,17176.0,19580.0,1265325.0,5174.0
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Aruba,7.0,855.0,1273.0,0.0,2119.75
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Australia,68.0,12179.0,16606.0,1024578.0,8709.666666666666
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Belgium,2.0,0.0,0.0,385473.0,5091.0
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Belize,119.0,16252.0,19815.0,6172.0,984.75
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Bermuda,1.0,0.0,0.0,5949.0,2103.0
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,"Bonaire, Sint Eustatius, and Saba",4.0,517.0,690.0,0.0,2127.0
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Brazil,88.0,18602.0,22164.0,1449796.0,5001.75
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Canada,584.0,67084.0,78749.0,742032.0,1532.1951219512196
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,Cayman Islands,12.0,1025.0,2130.0,0.0,1212.6666666666667


In [0]:
# ===================================================================
# OPTION B: Airport-level flights + distance to fruit fly counties
# Flights broken down by ORIGIN COUNTRY
# ===================================================================
from math import radians, cos, sin, asin, sqrt

# --- Haversine distance (miles) ---
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * 3956 * asin(sqrt(a))  # 3956 = Earth radius in miles

# --- County centroids ---
county_coords = {
    # Texas
    'Cameron County':    (26.15, -97.51),
    'Hidalgo County':    (26.40, -98.18),
    'Starr County':      (26.56, -98.74),
    'Webb County':       (27.76, -99.33),
    'Zapata County':     (26.99, -99.17),
    'Val Verde County':  (29.89, -101.15),
    'Willacy County':    (26.47, -97.59),
    # California
    'Alameda County':       (37.65, -121.89),
    'Contra Costa County':  (37.92, -121.95),
    'Fresno County':        (36.77, -119.72),
    'Kern County':          (35.35, -118.73),
    'Los Angeles County':   (34.05, -118.24),
    'Madera County':        (37.16, -119.82),
    'Merced County':        (37.19, -120.72),
    'Orange County':        (33.72, -117.78),
    'Placer County':        (39.06, -120.72),
    'Riverside County':     (33.74, -115.99),
    'Sacramento County':    (38.45, -121.35),
    'San Bernardino County':(34.84, -116.18),
    'San Diego County':     (32.82, -116.75),
    'San Joaquin County':   (37.93, -121.27),
    'San Mateo County':     (37.43, -122.35),
    'Santa Barbara County': (34.72, -120.03),
    'Santa Clara County':   (37.23, -121.69),
    'Sonoma County':        (38.53, -122.93),
    'Stanislaus County':    (37.56, -120.99),
    'Tulare County':        (36.21, -118.78),
    'Ventura County':       (34.36, -119.13),
    'Yolo County':          (38.69, -121.90),
    # Florida
    'Broward County':       (26.12, -80.14),
    'Hillsborough County':  (27.90, -82.36),
    'Miami-Dade County':    (25.76, -80.19),
    'Palm Beach County':    (26.65, -80.27),
    'Pinellas County':      (27.90, -82.73),
    'Seminole County':      (28.72, -81.24),
    'St. Lucie County':     (27.38, -80.39),
}

# --- 1. Aggregate flights at airport level + origin country ---
airport_agg = (
    df_us[df_us['state'].notna()]
    .groupby(['state', 'DEST', 'DEST_CITY_NAME', 'DEST_LAT', 'DEST_LON',
              'YEAR_WADS', 'MONTH_WADS', 'ORIGIN_COUNTRY_NAME'])
    .agg(airport_departures=('DEPARTURES_PERFORMED', 'sum'),
         airport_passengers=('PASSENGERS', 'sum'),
         airport_freight=('FREIGHT', 'sum'))
    .reset_index()
    .rename(columns={'YEAR_WADS': 'year', 'MONTH_WADS': 'month',
                     'DEST': 'airport_code', 'DEST_CITY_NAME': 'airport_city',
                     'DEST_LAT': 'airport_lat', 'DEST_LON': 'airport_lon',
                     'ORIGIN_COUNTRY_NAME': 'origin_country'})
)
print(f"Airport-level aggregation (by country): {airport_agg.shape[0]} rows")
print(f"  Unique origin countries: {airport_agg['origin_country'].nunique()}")

# --- 2. Add county coordinates ---
df_ff_coords = df_ff.copy()
df_ff_coords['county_lat'] = df_ff_coords['county'].map(lambda c: county_coords.get(c, (None,None))[0])
df_ff_coords['county_lon'] = df_ff_coords['county'].map(lambda c: county_coords.get(c, (None,None))[1])

missing_coords = df_ff_coords[df_ff_coords['county_lat'].isna()]['county'].unique()
if len(missing_coords) > 0:
    print(f"\n\u26a0\ufe0f Counties without coordinates: {list(missing_coords)}")
else:
    print("\n\u2705 All fruit fly counties have coordinates.")

# --- 3. Join and compute distances ---
RADIUS_MILES = 250

df_option_b = pd.merge(
    df_ff_coords, airport_agg,
    on=['state', 'year', 'month'],
    how='inner'
)

df_option_b['distance_miles'] = df_option_b.apply(
    lambda r: haversine(r['county_lat'], r['county_lon'],
                        r['airport_lat'], r['airport_lon'])
    if pd.notna(r['county_lat']) and pd.notna(r['airport_lat']) else None,
    axis=1
)

df_nearby = df_option_b[df_option_b['distance_miles'] <= RADIUS_MILES].copy()
df_nearby = df_nearby.sort_values(['state', 'county', 'year', 'month', 'distance_miles', 'origin_country'])
df_nearby = df_nearby.reset_index(drop=True)

print(f"\nBefore distance filter: {df_option_b.shape[0]} rows")
print(f"After {RADIUS_MILES}-mile filter:  {df_nearby.shape[0]} rows")

# --- 4. Save ---
out_path_b = os.path.join(current_dir, "option_b_county_airport_distance.parquet")
df_nearby.to_parquet(out_path_b, index=False)
print(f"\nSaved to {out_path_b}")

# Preview: Cameron County with country + airport breakdown (all columns, 50 rows)
print(f"\nSample \u2014 Cameron County, TX (Feb 2024, nearest airports by country):")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sample = df_nearby[
    (df_nearby['county'] == 'Cameron County') &
    (df_nearby['year'] == 2024) &
    (df_nearby['month'] == 2)
]
display(sample)

Airport-level aggregation (by country): 72907 rows
  Unique origin countries: 169

✅ All fruit fly counties have coordinates.

Before distance filter: 32470 rows
After 250-mile filter:  15411 rows

Saved to /Workspace/Users/c837609466@colostate.edu/Team_23/option_b_county_airport_distance.parquet

Sample — Cameron County, TX (Feb 2024, nearest airports by country):


state,county,year,month,fly_species,fly_detections,county_lat,county_lon,airport_code,airport_city,airport_lat,airport_lon,origin_country,airport_departures,airport_passengers,airport_freight,distance_miles
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,HRL,"Harlingen/San Benito, TX",26.2621,-97.77369,Mexico,1,42,0,18.076032591191996
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,MFE,"Mission/McAllen/Edinburg, TX",26.352233536029,-98.124349475225,Mexico,29,1452,0,40.52461063769204
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,CRP,"Corpus Christi, TX",27.79641,-97.40389,Canada,1,4,0,113.86409162613313
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,LRD,"Laredo, TX",27.53092,-99.50231,Bermuda,1,0,5949,155.4165763225977
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,LRD,"Laredo, TX",27.53092,-99.50231,Brazil,1,3,0,155.4165763225977
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,LRD,"Laredo, TX",27.53092,-99.50231,Mexico,23,0,92336,155.4165763225977
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,SAT,"San Antonio, TX",29.42458,-98.49461,El Salvador,1,16,0,233.95282401317604
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,SAT,"San Antonio, TX",29.42458,-98.49461,Guatemala,11,191,0,233.95282401317604
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,SAT,"San Antonio, TX",29.42458,-98.49461,Honduras,15,249,0,233.95282401317604
Texas,Cameron County,2024,2,Mexican Fruit Fly,2,26.15,-97.51,SAT,"San Antonio, TX",29.42458,-98.49461,Mexico,229,21841,0,233.95282401317604
